# Luồng xe theo trạm ở Jersey City

Tháng 1/2025. Đọc kết quả đã lưu hoặc chạy pipeline theo README để tạo SQLite.

In [1]:
from pathlib import Path
from contextlib import closing
import csv, json, sqlite3
from IPython.display import Markdown, display

root = Path.cwd()
if not (root / 'reports').is_dir():
    root = root.parent
reports = root / 'reports'
quality = json.loads((reports / 'quality.json').read_text(encoding='utf-8'))
print(json.dumps(quality, indent=2))
assert quality['raw_rows'] == sum(quality[k] for k in ('valid_rides','missing_ride_id','invalid_times','duplicate_ride_id'))

{
  "raw_rows": 105444,
  "valid_rides": 105444,
  "missing_ride_id": 0,
  "invalid_times": 0,
  "duplicate_ride_id": 0,
  "missing_start_id": 0,
  "missing_end_id": 298,
  "missing_or_invalid_coordinates": 0,
  "unknown_station_ids": 0,
  "departures": 26434,
  "arrivals": 26140
}


## Xếp hạng và so sánh

Xếp theo số ngày có net dương, tổng net dương, rồi mã trạm. Mở [bản đồ](../reports/map.html) để xem từng trạm.

In [2]:
display(Markdown((reports / 'findings.md').read_text(encoding='utf-8')))

# Luồng xe theo trạm ở Jersey City · tháng 1/2025

Chênh lệch = lượt đi trừ lượt đến. Giờ ưu tiên là giờ cao điểm có chênh lệch dương lặp lại nhiều ngày nhất.

| Trạm | Ngày dương | Tổng net dương | Giờ ưu tiên | Net/ngày trong tuần | Net/ngày cuối tuần |
|---|---:|---:|---|---:|---:|
| Grove St PATH | 23 | 851 | 18:00–19:00 | +14.13 | -1.13 |
| Bergen Ave & Sip Ave | 23 | 324 | 18:00–19:00 | +6.87 | +0.00 |
| Brunswick St | 23 | 196 | 08:00–09:00 | +4.65 | +1.00 |
| McGinley Square | 23 | 171 | 08:00–09:00 | +3.13 | +0.25 |
| Brunswick & 6th | 23 | 154 | 07:00–08:00 | +2.87 | -0.25 |

Ưu tiên kiểm tra **Grove St PATH** lúc **18:00–19:00**. Ở các ngày trong tuần có net dương trong giờ này, trạm có 445 lượt đi và 120 lượt đến.

Đã nạp 105,444 dòng từ 2 gói nguồn. Trong tháng 1, các trạm Jersey City có 26,434 lượt đi và 26,140 lượt đến.

So sánh theo giờ chia cho toàn bộ 23 ngày trong tuần hoặc 8 ngày cuối tuần, kể cả ngày có 0 lượt. Để xác nhận hết xe, cần chuỗi thời gian số xe sẵn có. Gói khu vực chỉ phản ánh các chuyến được ghi trong nguồn.

[Nguồn dữ liệu Citi Bike](https://citibikenyc.com/system-data)


## Kiểm tra một giờ bằng tay

Lấy một ô ngày–giờ ở trạm đứng đầu. Dùng số lượt đi trừ lượt đến, sau đó đối chiếu cả tháng với bảng xếp hạng.

In [3]:
def read_table(name):
    with (reports / name).open(encoding='utf-8-sig', newline='') as file:
        return list(csv.DictReader(file))

top = read_table('top5.csv')
detail = read_table('station_day_hour.csv')
for station in top:
    positive = [r for r in detail if r['station_id'] == station['station_id']
                and r['day_type'] == 'weekday' and int(r['hour']) in {7,8,9,16,17,18}
                and int(r['net']) > 0]
    assert len({r['day'] for r in positive}) == int(station['positive_days'])
    assert sum(int(r['net']) for r in positive) == int(station['positive_net'])
example = next(r for r in detail if r['station_id'] == top[0]['station_id'] and r['hour'] == '18')
assert int(example['departures']) - int(example['arrivals']) == int(example['net'])
print(example)
print('Verified all five stations against the day/hour table.')
db = reports / 'analysis.sqlite'
if db.exists():
    with closing(sqlite3.connect(db)) as con:
        actual = con.execute('SELECT SUM(departures), SUM(arrivals) FROM station_hour').fetchone()
    assert actual == (quality['departures'], quality['arrivals'])
    print('SQLite totals:', actual)

{'station_id': 'JC115', 'day': '2025-01-01', 'hour': '18', 'day_type': 'weekday', 'departures': '7', 'arrivals': '2', 'net': '5'}
Verified all five stations against the day/hour table.
SQLite totals: (26434, 26140)
